# Macro Regime + COT Positioning + Cross-Asset Tactical

    Fuses macro indicators (rates, inflation, growth), CFTC futures positioning (COT), futures/equity pricing and volatility context into a regime-aware tactical allocation view.

    **Category:** Multi-source macro / positioning / allocation

    **Primary API calls used:**
    - Macro series and indicators (FRED-heavy via macro domain or direct)
    - COT / positioning data (CFTC)
    - Futures and equity pricing (EOD / derivatives)
    - VIX term structure / vol regime (CBOE)

    Uses only public data surfaces available through the QuantJourney API + local pandas analytics. No user portfolios or IBOR required.


## Run Output

![70_macro_regime_cot_cross_asset](../plots/70_macro_regime_cot_cross_asset_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
import os
import json
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantjourney.sdk import QuantJourneyAPI

qj = QuantJourneyAPI.from_env()

START = os.getenv("QJ_EXAMPLE_START", "2018-01-01")
END = os.getenv("QJ_EXAMPLE_END", "2026-06-06")

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (13, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and "data" in payload:
        payload = payload["data"]
    if isinstance(payload, dict) and "value" in payload:
        return payload["value"]
    return payload


def safe_call(label: str, fn, **kwargs) -> Any:
    try:
        out = fn(**kwargs)
        print(f"{label}: ok")
        return out
    except Exception as exc:
        print(f"{label}: unavailable ({type(exc).__name__})")
        return None


def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ("rows", "data", "items", "prices", "results"):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


def price_frame(symbol: str, start: str = START, end: str = END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No price data for {symbol}")
    df["date"] = pd.to_datetime(df.get("date"))
    for col in ["open", "high", "low", "close", "adjusted_close", "volume"]:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    price_col = "adjusted_close" if df.get("adjusted_close", pd.Series()).notna().any() else "close"
    df["price"] = df[price_col]
    df = df.dropna(subset=["price"]).sort_values("date").set_index("date")
    return df


def get_macro_series(series_id: str, start: str = START, end: str = END) -> pd.Series:
    # Try common macro surfaces; fall back gracefully
    for meth in [
        lambda: qj.macro.series.get_time_series(series_id=series_id, start_date=start, end_date=end),
        lambda: qj.fred.get_fred_data(series_id=series_id, start_date=start, end_date=end),
    ]:
        try:
            payload = meth()
            rows = as_rows(payload)
            if rows:
                df = pd.DataFrame(rows)
                if "date" in df and "value" in df:
                    s = pd.to_numeric(df["value"], errors="coerce")
                    s.index = pd.to_datetime(df["date"])
                    return s.sort_index()
        except Exception:
            pass
    return pd.Series(dtype=float)


def get_cot(symbol: str) -> pd.DataFrame:
    payload = safe_call("COT " + symbol, getattr(qj, "cftc", qj).get_cot_data, symbol=symbol) or \
              safe_call("COT via commodities", lambda: qj.commodities.positioning.get_cot_data(symbol=symbol))
    return pd.DataFrame(as_rows(payload))


def simple_regime_label(macro_df: pd.DataFrame) -> pd.Series:
    # Very simple regime proxy using 10y yield trend + inflation proxy
    if "DGS10" in macro_df:
        yield_trend = macro_df["DGS10"].diff(63).rolling(21).mean()
    else:
        yield_trend = pd.Series(0, index=macro_df.index)
    regime = pd.Series("neutral", index=macro_df.index)
    regime[yield_trend > 0.15] = "rising_rates"
    regime[yield_trend < -0.15] = "falling_rates"
    return regime


In [ ]:
# === MULTI-SOURCE FUSION ===
print("Fetching macro, COT, futures/equity prices and vol context...")

# Macro proxies (adjust series ids to what your tenant has access to)
macro_ids = ["DGS10", "CPIAUCSL", "UNRATE", "FEDFUNDS"]
macro = {}
for sid in macro_ids:
    s = get_macro_series(sid)
    if not s.empty:
        macro[sid] = s
macro_df = pd.DataFrame(macro).sort_index()

# COT for a few key futures (example symbols; real COT symbols may differ)
cot_symbols = ["ES", "CL", "GC", "ZN"]  # equity index, crude, gold, 10y note
cot_data = {sym: get_cot(sym) for sym in cot_symbols}

# Cross asset price panel
cross_assets = ["SPY", "TLT", "GLD", "CL=F", "ES=F"]
prices = {}
for a in cross_assets:
    try:
        df = price_frame(a)
        prices[a] = df["price"]
    except Exception as e:
        print(f"price {a}: {e}")
px = pd.DataFrame(prices).dropna(how="all")
ret = px.pct_change().dropna()

# Simple regime
reg = simple_regime_label(macro_df.reindex(px.index, method="ffill"))

# Aggregate COT net positioning (very rough example)
cot_net = pd.DataFrame({k: v.get("net_positions", pd.Series()) for k, v in cot_data.items() if isinstance(v, pd.DataFrame)})
if not cot_net.empty:
    cot_z = (cot_net - cot_net.rolling(52).mean()) / cot_net.rolling(52).std()

print("Regime counts:\n", reg.value_counts())

# Regime-conditional performance
perf = {}
for r in reg.unique():
    mask = reg == r
    if mask.sum() > 20:
        perf[r] = (ret[mask].mean() * 252).round(3)
print("\nRegime-conditional annualized returns (approx):\n", pd.DataFrame(perf))

# Quick visual
fig, ax = plt.subplots(1, 1)
for col in ret.columns[:4]:
    (1 + ret[col]).cumprod().plot(ax=ax, label=col, alpha=0.8)
ax.set_title("Cross-asset cumulative (colored by regime in full version)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nThis candidate demonstrates fusion of macro + COT positioning + multi-asset prices.\nIn a full version one would map COT z-scores to tactical tilts per regime and show risk contribution.")


## Notes

This is a candidate workflow using only public multi-source data available via the QuantJourney API.
No PMS, IBOR or user portfolio data is used — everything is constructed from macro series, COT, futures/equity pricing and volatility surfaces.
In production retain request_ids, respect tenant scopes and treat COT symbol mapping as configuration.